# 04 - LOSO Tabular Baselines (Physio + EEG)

This notebook implements robust tabular baselines using the built fusion dataset:

1. Strict LOSO (leave-one-subject-out) evaluation
2. Leakage-safe fold preprocessing (fit on train only)
3. Baselines: Logistic Regression, Random Forest, XGBoost (if available)
4. Macro-F1 / Balanced Accuracy / Accuracy tracking
5. Per-fold and aggregate result exports

In [1]:
# SECTION 1: Imports
import json
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import f1_score, balanced_accuracy_score, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Optional XGBoost baseline
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

print('✓ Imports loaded | XGBoost available:', HAS_XGB)

✓ Imports loaded | XGBoost available: True


In [2]:
# SECTION 2: Configuration and Paths
@dataclass
class TrainConfig:
    output_version: str = 'v1_loso_tabular'
    max_folds: int = 5  # Dev mode: evaluate 5 representative LOSO folds
    random_state: int = 42
    drop_leakage_prefixes: tuple = ('ID__', 'Repetition__')
    fold_selection_mode: str = 'fusion_representative'  # fusion_representative | first_n

CFG = TrainConfig()

# Detect runtime
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    try:
        drive.mount('/content/drive', force_remount=False)
    except Exception:
        pass
    ROOT = Path('/content/drive/MyDrive')
else:
    ROOT = Path.home() / 'Desktop' / 'thesis'

FUSION_DIR = ROOT / 'research_outputs' / 'fusion' / 'v1_fusion'
FUSION_PATH = FUSION_DIR / 'fusion_dataset.csv'
OUT_DIR = ROOT / 'research_outputs' / 'fusion_training' / CFG.output_version
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('✓ Paths configured')
print('  FUSION_PATH:', FUSION_PATH)
print('  OUT_DIR:', OUT_DIR)
print('  fold_selection_mode:', CFG.fold_selection_mode)
print('  max_folds:', CFG.max_folds)

Mounted at /content/drive
✓ Paths configured
  FUSION_PATH: /content/drive/MyDrive/research_outputs/fusion/v1_fusion/fusion_dataset.csv
  OUT_DIR: /content/drive/MyDrive/research_outputs/fusion_training/v1_loso_tabular
  fold_selection_mode: fusion_representative
  max_folds: 5


In [3]:
# SECTION 3: Load Dataset and Prepare Feature Space
if not FUSION_PATH.exists():
    raise FileNotFoundError(f'Fusion dataset not found: {FUSION_PATH}')

df = pd.read_csv(FUSION_PATH)
print('Loaded fusion dataset:', df.shape)

# Label column selection
label_col = 'label' if 'label' in df.columns else 'pseudo_label'
if label_col not in df.columns:
    raise ValueError('No label column found (expected label or pseudo_label).')

# Keys and metadata columns excluded from features
meta_cols = {
    'subject_id', 'task_name', 'task_file', 'split', 'window_idx',
    'start_idx', 'end_idx', 'n_samples',
    'has_physio', 'has_eeg', 'has_speech', 'has_au', 'has_nlp', 'n_modalities_present',
    'label', 'pseudo_label'
}

feature_cols = [c for c in df.columns if c not in meta_cols]
# Drop known leakage-like metadata-derived features
feature_cols = [
    c for c in feature_cols
    if not any(c.startswith(pref) for pref in CFG.drop_leakage_prefixes)
]

if 'subject_id' not in df.columns:
    raise ValueError('subject_id column missing; LOSO cannot be built.')

# Keep only numeric model features. Some joined tables may carry string columns
# (for example prefixed split/task fields) that should never enter estimators.
X_raw = df[feature_cols].copy()
numeric_feature_cols = X_raw.select_dtypes(include=[np.number, 'bool']).columns.tolist()
X = X_raw[numeric_feature_cols].copy()

y_raw = df[label_col].astype(str).fillna('NA')
subjects = df['subject_id'].astype(str).values

le = LabelEncoder()
y = le.fit_transform(y_raw)

print('✓ Prepared training matrix')
print('  Features before numeric filter:', len(feature_cols))
print('  Features after  numeric filter:', X.shape[1])
print('  Samples:', X.shape[0])
print('  Classes:', list(le.classes_))
print('  Subjects:', len(np.unique(subjects)))

Loaded fusion dataset: (5640, 257)
✓ Prepared training matrix
  Features before numeric filter: 226
  Features after  numeric filter: 225
  Samples: 5640
  Classes: ['cognitive_load', 'high_stress', 'industrial_task', 'low_load', 'other']
  Subjects: 52


In [4]:
# SECTION 4: Build Models
def make_models(n_classes: int, random_state: int = 42) -> Dict[str, object]:
    models = {
        'logreg': LogisticRegression(
            max_iter=2000,
            class_weight='balanced',
            multi_class='multinomial',
            solver='lbfgs',
            n_jobs=None
        ),
        'rf': RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            class_weight='balanced_subsample',
            random_state=random_state,
            n_jobs=-1
        )
    }

    if HAS_XGB:
        models['xgb'] = XGBClassifier(
            n_estimators=600,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective='multi:softprob',
            num_class=n_classes,
            tree_method='hist',
            eval_metric='mlogloss',
            random_state=random_state
        )

    return models

models = make_models(n_classes=len(le.classes_), random_state=CFG.random_state)
print('✓ Models:', list(models.keys()))

✓ Models: ['logreg', 'rf', 'xgb']


In [5]:
# SECTION 5: LOSO Training Utilities (Run Fold-by-Fold)

def select_fusion_representative_subjects(df: pd.DataFrame, n_folds: int) -> List[str]:
    """Pick subjects that best cover multimodal HRC task families.

    Families: industrial, cognitive, stress, recovery.
    We rank subjects by family coverage first, then by number of rows.
    """
    work = df[['subject_id', 'task_name', 'has_physio', 'has_eeg']].copy()
    work['subject_id'] = work['subject_id'].astype(str)
    work['task_name'] = work['task_name'].astype(str).str.lower()

    def family_flags(task_name: str) -> Dict[str, int]:
        return {
            'industrial': int(('cobot-task' in task_name) or ('manual-task' in task_name)),
            'cognitive': int(any(k in task_name for k in ['hanoi', 'n-back', 'stroop', 'mat'])),
            'stress': int('vr-plank' in task_name),
            'recovery': int(any(k in task_name for k in ['rest', 'meditation'])),
        }

    fam_df = work['task_name'].apply(family_flags).apply(pd.Series)
    prof = pd.concat([work[['subject_id', 'has_physio', 'has_eeg']], fam_df], axis=1)

    sub_prof = (
        prof.groupby('subject_id', as_index=False)
        .agg(
            has_physio=('has_physio', 'max'),
            has_eeg=('has_eeg', 'max'),
            industrial=('industrial', 'max'),
            cognitive=('cognitive', 'max'),
            stress=('stress', 'max'),
            recovery=('recovery', 'max'),
            n_rows=('subject_id', 'size'),
        )
    )

    sub_prof['family_coverage'] = sub_prof[['industrial', 'cognitive', 'stress', 'recovery']].sum(axis=1)
    sub_prof['multimodal_ready'] = ((sub_prof['has_physio'] > 0) & (sub_prof['has_eeg'] > 0)).astype(int)

    sub_prof = sub_prof.sort_values(
        ['multimodal_ready', 'family_coverage', 'n_rows', 'subject_id'],
        ascending=[False, False, False, True]
    ).reset_index(drop=True)

    chosen = sub_prof['subject_id'].head(n_folds).tolist()
    print('Selected representative subjects (fusion_representative):', chosen)
    return chosen


all_subjects = sorted(np.unique(subjects))
if CFG.max_folds > 0:
    if getattr(CFG, 'fold_selection_mode', 'first_n') == 'fusion_representative':
        selected_subjects = select_fusion_representative_subjects(df, CFG.max_folds)
    else:
        selected_subjects = all_subjects[:CFG.max_folds]
else:
    selected_subjects = all_subjects

print(f'Prepared fold subjects ({len(selected_subjects)}): {selected_subjects}')

FOLD_CACHE_DIR = OUT_DIR / 'fold_cache'
FOLD_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print('Fold cache dir:', FOLD_CACHE_DIR)


def run_one_fold(fold_index_1based: int):
    if fold_index_1based < 1 or fold_index_1based > len(selected_subjects):
        raise ValueError(f'fold_index must be in [1, {len(selected_subjects)}]')

    sid = selected_subjects[fold_index_1based - 1]
    test_mask = subjects == sid
    train_mask = ~test_mask

    X_train, X_test = X.loc[train_mask], X.loc[test_mask]
    y_train, y_test = y[train_mask], y[test_mask]

    pre = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    X_train_p = pre.fit_transform(X_train)
    X_test_p = pre.transform(X_test)

    fold_rows = []
    oof_rows = []

    for model_name, model in models.items():
        model.fit(X_train_p, y_train)
        y_pred = model.predict(X_test_p)

        f1m = f1_score(y_test, y_pred, average='macro')
        bacc = balanced_accuracy_score(y_test, y_pred)
        acc = accuracy_score(y_test, y_pred)

        fold_rows.append({
            'fold': fold_index_1based,
            'test_subject': sid,
            'model': model_name,
            'n_train': int(train_mask.sum()),
            'n_test': int(test_mask.sum()),
            'macro_f1': float(f1m),
            'balanced_acc': float(bacc),
            'accuracy': float(acc),
        })

        for idx_local, pred in zip(np.where(test_mask)[0], y_pred):
            oof_rows.append({
                'row_idx': int(idx_local),
                'subject_id': str(subjects[idx_local]),
                'model': model_name,
                'y_true': int(y[idx_local]),
                'y_pred': int(pred),
            })

    fold_df_local = pd.DataFrame(fold_rows)
    oof_df_local = pd.DataFrame(oof_rows)

    fold_metrics_path = FOLD_CACHE_DIR / f'fold_{fold_index_1based:02d}_metrics.csv'
    fold_oof_path = FOLD_CACHE_DIR / f'fold_{fold_index_1based:02d}_oof.csv'
    fold_df_local.to_csv(fold_metrics_path, index=False)
    oof_df_local.to_csv(fold_oof_path, index=False)

    print(f'✓ Completed fold {fold_index_1based}/{len(selected_subjects)} | subject={sid}')
    print('  -', fold_metrics_path)
    print('  -', fold_oof_path)
    display(fold_df_local)


print('Use the next cells to run folds individually (safe for Colab disconnects).')

Selected representative subjects (fusion_representative): ['10000', '1032', '1038', '1065', '1035']
Prepared fold subjects (5): ['10000', '1032', '1038', '1065', '1035']
Fold cache dir: /content/drive/MyDrive/research_outputs/fusion_training/v1_loso_tabular/fold_cache
Use the next cells to run folds individually (safe for Colab disconnects).


In [12]:
# SECTION 6: Aggregate Results and Export

# Collect completed fold artifacts from cache (works after reconnects).
fold_metric_files = sorted(FOLD_CACHE_DIR.glob('fold_*_metrics.csv')) if 'FOLD_CACHE_DIR' in globals() else []
fold_oof_files = sorted(FOLD_CACHE_DIR.glob('fold_*_oof.csv')) if 'FOLD_CACHE_DIR' in globals() else []

if not fold_metric_files:
    raise ValueError('No fold metric files found. Run at least one fold cell in SECTION 5B.')

fold_df = pd.concat([pd.read_csv(p) for p in fold_metric_files], ignore_index=True)
oof_df = pd.concat([pd.read_csv(p) for p in fold_oof_files], ignore_index=True) if fold_oof_files else pd.DataFrame()

agg_df = (
    fold_df.groupby('model', as_index=False)
    .agg(
        macro_f1_mean=('macro_f1', 'mean'),
        macro_f1_std=('macro_f1', 'std'),
        balanced_acc_mean=('balanced_acc', 'mean'),
        accuracy_mean=('accuracy', 'mean'),
        folds=('fold', 'count')
    )
    .sort_values('macro_f1_mean', ascending=False)
    .reset_index(drop=True)
)

fold_path = OUT_DIR / 'loso_fold_metrics.csv'
agg_path = OUT_DIR / 'loso_aggregate_metrics.csv'
oof_path = OUT_DIR / 'loso_oof_predictions.csv'
manifest_path = OUT_DIR / 'run_manifest.json'

fold_df.to_csv(fold_path, index=False)
agg_df.to_csv(agg_path, index=False)
oof_df.to_csv(oof_path, index=False)

manifest = {
    'config': asdict(CFG),
    'fusion_path': str(FUSION_PATH),
    'n_samples': int(len(df)),
    'n_features': int(X.shape[1]) if 'X' in globals() else None,
    'classes': list(map(str, le.classes_)),
    'models': list(models.keys()),
    'n_completed_folds': int(len(fold_metric_files)),
    'completed_fold_files': [str(p) for p in fold_metric_files],
    'outputs': {
        'fold_metrics': str(fold_path),
        'aggregate_metrics': str(agg_path),
        'oof_predictions': str(oof_path)
    }
}

with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

print('✓ Exported baseline outputs')
print('  completed folds:', len(fold_metric_files))
print('  -', fold_path)
print('  -', agg_path)
print('  -', oof_path)
print('  -', manifest_path)

display(agg_df)

✓ Exported baseline outputs
  completed folds: 5
  - /content/drive/MyDrive/research_outputs/fusion_training/v1_loso_tabular/loso_fold_metrics.csv
  - /content/drive/MyDrive/research_outputs/fusion_training/v1_loso_tabular/loso_aggregate_metrics.csv
  - /content/drive/MyDrive/research_outputs/fusion_training/v1_loso_tabular/loso_oof_predictions.csv
  - /content/drive/MyDrive/research_outputs/fusion_training/v1_loso_tabular/run_manifest.json


,model,macro_f1_mean,macro_f1_std,balanced_acc_mean,accuracy_mean,folds
0,xgb,0.616763,0.175761,0.652399,0.721014,5
1,rf,0.567677,0.048816,0.606493,0.722459,5
2,logreg,0.408679,0.132284,0.546144,0.532201,5


## SECTION 5B: Run LOSO Folds Individually

Run Cells 8 to 12 independently. Each cell writes fold artifacts to `fold_cache`, so reconnecting Colab does not lose completed folds.

In [7]:
# Fold 1
run_one_fold(1)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


✓ Completed fold 1/5 | subject=10000
  - /content/drive/MyDrive/research_outputs/fusion_training/v1_loso_tabular/fold_cache/fold_01_metrics.csv
  - /content/drive/MyDrive/research_outputs/fusion_training/v1_loso_tabular/fold_cache/fold_01_oof.csv


,fold,test_subject,model,n_train,n_test,macro_f1,balanced_acc,accuracy
0,1,10000,logreg,5477,163,0.388662,0.466950,0.527607
1,1,10000,rf,5477,163,0.538797,0.512412,0.754601
2,1,10000,xgb,5477,163,0.631905,0.624467,0.809816


In [8]:
# Fold 2
run_one_fold(2)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


✓ Completed fold 2/5 | subject=1032
  - /content/drive/MyDrive/research_outputs/fusion_training/v1_loso_tabular/fold_cache/fold_02_metrics.csv
  - /content/drive/MyDrive/research_outputs/fusion_training/v1_loso_tabular/fold_cache/fold_02_oof.csv


,fold,test_subject,model,n_train,n_test,macro_f1,balanced_acc,accuracy
0,2,1032,logreg,5477,163,0.240317,0.358756,0.509202
1,2,1032,rf,5477,163,0.532014,0.549912,0.828221
2,2,1032,xgb,5477,163,0.453709,0.490073,0.803681


In [9]:
# Fold 3
run_one_fold(3)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


✓ Completed fold 3/5 | subject=1038
  - /content/drive/MyDrive/research_outputs/fusion_training/v1_loso_tabular/fold_cache/fold_03_metrics.csv
  - /content/drive/MyDrive/research_outputs/fusion_training/v1_loso_tabular/fold_cache/fold_03_oof.csv


,fold,test_subject,model,n_train,n_test,macro_f1,balanced_acc,accuracy
0,3,1038,logreg,5480,160,0.611515,0.798434,0.79375
1,3,1038,rf,5480,160,0.626991,0.772243,0.85625
2,3,1038,xgb,5480,160,0.792817,0.838693,0.82500


In [10]:
# Fold 4
run_one_fold(4)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


✓ Completed fold 4/5 | subject=1065
  - /content/drive/MyDrive/research_outputs/fusion_training/v1_loso_tabular/fold_cache/fold_04_metrics.csv
  - /content/drive/MyDrive/research_outputs/fusion_training/v1_loso_tabular/fold_cache/fold_04_oof.csv


,fold,test_subject,model,n_train,n_test,macro_f1,balanced_acc,accuracy
0,4,1065,logreg,5482,158,0.400155,0.501316,0.620253
1,4,1065,rf,5482,158,0.614420,0.599785,0.803797
2,4,1065,xgb,5482,158,0.783469,0.768369,0.848101


In [11]:
# Fold 5
run_one_fold(5)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


✓ Completed fold 5/5 | subject=1035
  - /content/drive/MyDrive/research_outputs/fusion_training/v1_loso_tabular/fold_cache/fold_05_metrics.csv
  - /content/drive/MyDrive/research_outputs/fusion_training/v1_loso_tabular/fold_cache/fold_05_oof.csv


,fold,test_subject,model,n_train,n_test,macro_f1,balanced_acc,accuracy
0,5,1035,logreg,5483,157,0.402746,0.605263,0.210191
1,5,1035,rf,5483,157,0.526165,0.598112,0.369427
2,5,1035,xgb,5483,157,0.421915,0.540390,0.318471


In [14]:
# SECTION 7: Compact Achieved Results Summary

print('=== ACHIEVED RESULTS SUMMARY ===')

if 'selected_subjects' in globals():
    print('selected_subjects:', selected_subjects)

if 'fold_metric_files' in globals():
    print('completed_fold_files:', len(fold_metric_files))

if 'fold_df' in globals() and not fold_df.empty:
    print('fold_df shape:', fold_df.shape)
    print('models:', sorted(fold_df['model'].unique().tolist()))
    print('test_subjects:', sorted(fold_df['test_subject'].astype(str).unique().tolist()))
    print('rows per model:')
    print(fold_df['model'].value_counts().to_string())

if 'agg_df' in globals() and not agg_df.empty:
    print('\nAggregate metrics (sorted by macro_f1_mean):')
    print(agg_df.to_string(index=False))

    best = agg_df.iloc[0]
    print('\nBest model:')
    print(f"  model={best['model']}")
    print(f"  macro_f1_mean={best['macro_f1_mean']:.4f} ± {best['macro_f1_std']:.4f}")
    print(f"  balanced_acc_mean={best['balanced_acc_mean']:.4f}")
    print(f"  accuracy_mean={best['accuracy_mean']:.4f}")

if 'manifest_path' in globals():
    print('\nmanifest_path:', manifest_path)
if 'fold_path' in globals():
    print('fold_metrics_path:', fold_path)
if 'agg_path' in globals():
    print('aggregate_metrics_path:', agg_path)


=== ACHIEVED RESULTS SUMMARY ===
selected_subjects: ['10000', '1032', '1038', '1065', '1035']
completed_fold_files: 5
fold_df shape: (15, 8)
models: ['logreg', 'rf', 'xgb']
test_subjects: ['10000', '1032', '1035', '1038', '1065']
rows per model:
model
logreg    5
rf        5
xgb       5

Aggregate metrics (sorted by macro_f1_mean):
 model  macro_f1_mean  macro_f1_std  balanced_acc_mean  accuracy_mean  folds
   xgb       0.616763      0.175761           0.652399       0.721014      5
    rf       0.567677      0.048816           0.606493       0.722459      5
logreg       0.408679      0.132284           0.546144       0.532201      5

Best model:
  model=xgb
  macro_f1_mean=0.6168 ± 0.1758
  balanced_acc_mean=0.6524
  accuracy_mean=0.7210

manifest_path: /content/drive/MyDrive/research_outputs/fusion_training/v1_loso_tabular/run_manifest.json
fold_metrics_path: /content/drive/MyDrive/research_outputs/fusion_training/v1_loso_tabular/loso_fold_metrics.csv
aggregate_metrics_path: /content